#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

# Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
  df
  .withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
    .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
    .otherwise("n/a")

    )
  .withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
    .when(F.upper(F.col("cst_gndr")) == "M", "Male")
    .otherwise("n/a")

    )
)


## Renaming the Colums

In [0]:
for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

In [0]:
#trim the string
#normalization for material_status, gndr
#names are not friendly

df.display()

# Write Into Silver Table

In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("workspace.silver.crm_customers")
)

In [0]:
%sql
select * from workspace.silver.crm_customers